# Media Framing Batch API Preparation

This notebook prepares the final thesis batch input for the media-framing codebook.

It keeps the Katinka-style context-unit logic:
- one request per merged context window
- multiple media hits inside one window stay in the same row via `hit_text`
- `row_id` is preserved end-to-end
- Tagesschau is included, but direct Tagesschau self-references are removed

The notebook does not submit anything unless you explicitly set the final flag to `True`.


In [8]:
from __future__ import annotations

import json
import sys
import tempfile
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd() / "2a_NER",
    Path.cwd(),
    Path.cwd().parent / "2a_NER",
    Path("/Users/MattisHaumann/Dev/Thesis/2a_NER"),
]
NOTEBOOK_DIR = next(
    (path for path in NOTEBOOK_DIR_CANDIDATES if (path / "media_framing_batch_utils.py").exists()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Could not locate 2a_NER/media_framing_batch_utils.py from the current working directory.")

PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from media_framing_batch_utils import (
    build_batch_requests_df,
    compile_master_pattern,
    create_batch_job,
    download_openai_file,
    estimate_batch_cost,
    extract_media_contexts,
    parse_batch_output_file,
    read_env_value,
    split_batch_requests_by_estimated_tokens,
    upload_batch_file,
    validate_batch_requests,
    write_batch_jsonl,
    write_manifest_csv,
)

DATA_PATH = NOTEBOOK_DIR / "df_combined.csv"
PROMPT_PATH = NOTEBOOK_DIR / "framing_codebook_prompt.txt"
OUTPUT_ROOT = NOTEBOOK_DIR / "outputs" / "batch_media_framing"
OUTPUT_DIR = OUTPUT_ROOT / "final_thesis"
LEGACY_OUTPUT_DIR = OUTPUT_ROOT / "legacy_batches"
FULL_BATCH_JSONL_PATH = OUTPUT_DIR / "media_framing_thesis_batch.jsonl"
FULL_MANIFEST_PATH = OUTPUT_DIR / "media_framing_thesis_manifest.csv"
FULL_RESULTS_PATH = OUTPUT_DIR / "media_framing_thesis_results.csv"
FULL_ERRORS_PATH = OUTPUT_DIR / "media_framing_thesis_errors.csv"
BATCH_JOB_PATH = OUTPUT_DIR / "media_framing_thesis_batch_job.json"
EXISTING_MANIFEST_PATH = LEGACY_OUTPUT_DIR / "media_framing_full_manifest.csv"
PART_JSONL_PATHS = []
PART_MANIFEST_PATHS = []
UPLOAD_BATCH_JSONL_PATH = FULL_BATCH_JSONL_PATH
UPLOAD_MANIFEST_PATH = FULL_MANIFEST_PATH

MODEL_NAME = "gpt-5-mini"
WINDOW = 1
SAFE_BATCH_INPUT_TOKEN_LIMIT = 4_500_000  # Tier-1-safe default for gpt-5-mini under a 5,000,000 enqueued-token limit.

FRAME_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "category": {
            "type": "string",
            "enum": [
                "POSITIONS-/PARTEILICHKEITS-BIAS",
                "VERZERRUNG/MANIPULATION",
                "DISINFORMATION/FALSCHDARSTELLUNG",
                "VERSAGEN/INKOMPETENZ",
                "NEUTRAL",
                "IRRELEVANT",
            ],
        },
        "evidence": {"type": "string"},
    },
    "required": ["category", "evidence"],
}

ANALYSIS_INSTRUCTIONS = (
    "Return valid JSON that matches the schema exactly. "
    "Do not add any keys beyond category and evidence."
)

LEGACY_RESULT_COLUMNS = [
    "hit_id",
    "row_id",
    "source",
    "Title",
    "hit_text",
    "context_idx",
    "count_hits",
    "count_unique_entities",
    "context_window",
    "model",
    "response_id",
    "category",
    "evidence",
    "raw_response_json",
]

for required_path in [DATA_PATH, PROMPT_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file not found: {required_path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Prompt path: {PROMPT_PATH}")
print(f"Batch JSONL path: {FULL_BATCH_JSONL_PATH}")
print(f"Manifest path: {FULL_MANIFEST_PATH}")


Data path: /Users/MattisHaumann/Dev/Thesis/2a_NER/df_combined.csv
Prompt path: /Users/MattisHaumann/Dev/Thesis/2a_NER/framing_codebook_prompt.txt
Batch JSONL path: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_batch.jsonl
Manifest path: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_manifest.csv


## 1. Load the Combined Data and the Codebook Prompt

In [9]:
df = pd.read_csv(DATA_PATH)

if "row_id" not in df.columns:
    df = df.reset_index().rename(columns={"index": "row_id"})

required_columns = {"row_id", "source", "Title", "Text"}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"df_combined.csv is missing required columns: {missing_columns}")

search_columns = [column for column in ["row_id", "source", "Date", "Title", "Text"] if column in df.columns]
search_df = df[search_columns].copy()
search_df["source"] = search_df["source"].fillna("").astype(str)

CODEBOOK_PROMPT = PROMPT_PATH.read_text(encoding="utf-8").strip()
if not CODEBOOK_PROMPT:
    raise ValueError(f"Prompt file is empty: {PROMPT_PATH}")

print(f"Articles loaded: {len(search_df):,}")
print(f"Unique sources: {search_df['source'].nunique():,}")
display(search_df.head(3))


Articles loaded: 20,440
Unique sources: 7


,row_id,source,Date,Title,Text
0,1,Antispiegel,2025-08-01,Bereitet der Westen die Entmachtung oder sogar...,Der Streit um das Nationale Anti-Korruptionsbü...
1,2,Antispiegel,2025-08-01,EU-Kommission hat Textnachrichten zum Kauf der...,Die New York Times versucht seit langem vor Ge...
2,3,Antispiegel,2025-08-01,Wahlkommission rechtfertigt Einmischung der EU...,Ende September stehen in Moldawien Parlamentsw...


## 2. Extract Context Windows and Run Thesis-Facing Checks

This is the key preparation step. It applies the mainstream-media filter, keeps Tagesschau in the data, and removes only direct Tagesschau self-references.


In [10]:
MASTER_PATTERN = compile_master_pattern()

extraction = extract_media_contexts(search_df, pattern=MASTER_PATTERN, window=WINDOW)
media_context_df = extraction["media_context_df"].copy()
media_article_df = extraction["media_article_df"].copy()
kept_hits_df = extraction["kept_hits_df"].copy()
excluded_hits_df = extraction["excluded_hits_df"].copy()

required_context_columns = [
    "row_id",
    "source",
    "Title",
    "hit_text",
    "context_idx",
    "count_hits",
    "count_unique_entities",
    "context_window",
]
missing_context_columns = [column for column in required_context_columns if column not in media_context_df.columns]
if missing_context_columns:
    raise ValueError(f"media_context_df is missing required columns: {missing_context_columns}")
if media_context_df.empty:
    raise ValueError("media_context_df is empty after extraction.")
if media_context_df["row_id"].isna().any():
    raise AssertionError("row_id is missing in media_context_df.")
if media_context_df.duplicated(["row_id", "context_idx"]).any():
    raise AssertionError("row_id/context_idx pairs must be unique.")
if media_context_df["hit_text"].fillna("").eq("").any():
    raise AssertionError("Empty hit_text values found in media_context_df.")

self_term_pattern = r"(^| \| )(Tagesschau|Tagesschau24|ARD|Das Erste|das Erste)( \| |$)"
tagesschau_self_leaks_df = media_context_df.loc[
    (media_context_df["source"] == "Tagesschau")
    & (media_context_df["hit_text"].str.contains(self_term_pattern, regex=True, na=False)),
    ["row_id", "Title", "hit_text", "context_window"],
].copy()
if not tagesschau_self_leaks_df.empty:
    raise AssertionError("Tagesschau self-references still leaked into the final context rows.")

outlet_summary_df = (
    media_context_df.groupby("source", as_index=False)
    .agg(
        context_rows=("row_id", "size"),
        unique_articles=("row_id", "nunique"),
        multi_hit_windows=("count_unique_entities", lambda values: int((values > 1).sum())),
    )
    .sort_values(["context_rows", "unique_articles", "source"], ascending=[False, False, True])
    .reset_index(drop=True)
)

comparison_summary_df = pd.DataFrame()
changed_rows_df = pd.DataFrame()
if EXISTING_MANIFEST_PATH.exists():
    existing_manifest_df = pd.read_csv(EXISTING_MANIFEST_PATH)
    comparison_summary_df = pd.DataFrame(
        [
            {"metric": "existing_manifest_rows", "value": len(existing_manifest_df)},
            {"metric": "new_context_rows", "value": len(media_context_df)},
            {"metric": "row_delta", "value": len(media_context_df) - len(existing_manifest_df)},
            {
                "metric": "existing_tagesschau_rows",
                "value": int((existing_manifest_df["source"] == "Tagesschau").sum()),
            },
            {
                "metric": "new_tagesschau_rows",
                "value": int((media_context_df["source"] == "Tagesschau").sum()),
            },
        ]
    )

    changed_rows_df = existing_manifest_df[
        ["row_id", "context_idx", "source", "hit_text", "count_unique_entities"]
    ].merge(
        media_context_df[
            ["row_id", "context_idx", "source", "hit_text", "count_unique_entities"]
        ],
        on=["row_id", "context_idx", "source"],
        how="outer",
        suffixes=("_old", "_new"),
        indicator=True,
    )
    changed_rows_df = changed_rows_df.loc[
        (changed_rows_df["_merge"] != "both")
        | (changed_rows_df["hit_text_old"].fillna("") != changed_rows_df["hit_text_new"].fillna(""))
        | (
            changed_rows_df["count_unique_entities_old"].fillna(-1)
            != changed_rows_df["count_unique_entities_new"].fillna(-1)
        )
    ].reset_index(drop=True)

print(f"Context rows: {len(media_context_df):,}")
print(f"Unique filtered articles: {len(media_article_df):,}")
print(f"Excluded Tagesschau self-hits: {len(excluded_hits_df.loc[excluded_hits_df['exclusion_reason'] == 'tagesschau_self_reference']):,}")
display(outlet_summary_df)
if not comparison_summary_df.empty:
    display(comparison_summary_df)
if not changed_rows_df.empty:
    display(changed_rows_df.head(20))


Context rows: 11,975
Unique filtered articles: 7,204
Excluded Tagesschau self-hits: 7,213


/var/folders/_q/1md1f9cs5rqdj5k3g3mxp5240000gp/T/ipykernel_12982/2543771931.py:34: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & (media_context_df["hit_text"].str.contains(self_term_pattern, regex=True, na=False)),


,source,context_rows,unique_articles,multi_hit_windows
0,Tagesschau,4370,2972,339
1,Tichys_Einblick,2282,1107,503
2,Nius,2026,1105,526
3,RT_de,1864,1149,313
4,Antispiegel,540,221,29
5,Compact,512,356,66
6,Deutschlandkurier,381,294,88


,metric,value
0,existing_manifest_rows,12009
1,new_context_rows,11975
2,row_delta,-34
3,existing_tagesschau_rows,4404
4,new_tagesschau_rows,4370


,row_id,context_idx,source,hit_text_old,count_unique_entities_old,hit_text_new,count_unique_entities_new,_merge
0,94,1,Antispiegel,Redaktionsnetzwerk Deutschland,1,RND,1.0,both
1,115,1,Antispiegel,das Erste,1,Das Erste,1.0,both
2,136,5,Antispiegel,Süddeutschen Zeitung | Süddeutsche Zeitung | T...,3,Süddeutsche Zeitung | Tagesschau,2.0,both
3,144,1,Antispiegel,das Erste,1,Das Erste,1.0,both
4,196,1,Antispiegel,Rheinischen Post,1,Rheinische Post,1.0,both
5,271,2,Antispiegel,Redaktionsnetzwerk Deutschland,1,RND,1.0,both
6,291,1,Antispiegel,Spiegel | Süddeutschen Zeitung,2,Spiegel | Süddeutsche Zeitung,2.0,both
7,348,2,Antispiegel,das Erste,1,Das Erste,1.0,both
8,388,3,Antispiegel,das Erste,1,Das Erste,1.0,both
9,496,1,Antispiegel,das Erste,1,Das Erste,1.0,both


## 3. Build the Batch Requests and Run a Parser Smoke Test

This check makes sure the batch output can later be converted back into the same result-row structure as the earlier synchronous GPT run.


In [11]:
batch_requests_df = build_batch_requests_df(
    media_context_df,
    prompt_template=CODEBOOK_PROMPT,
    model_name=MODEL_NAME,
    analysis_instructions=ANALYSIS_INSTRUCTIONS,
    frame_schema=FRAME_SCHEMA,
)

validation_errors = validate_batch_requests(batch_requests_df)
if validation_errors:
    raise ValueError("Batch validation failed:\n" + "\n".join(validation_errors[:20]))

manifest_df = batch_requests_df[
    [
        "custom_id",
        "hit_id",
        "row_id",
        "source",
        "Title",
        "hit_text",
        "context_idx",
        "count_hits",
        "count_unique_entities",
        "context_window",
    ]
].copy()

if manifest_df["row_id"].isna().any():
    raise AssertionError("row_id is missing in the manifest.")
if manifest_df["hit_id"].duplicated().any():
    raise AssertionError("hit_id values are not unique.")
if manifest_df["custom_id"].duplicated().any():
    raise AssertionError("custom_id values are not unique.")

smoke_record = {
    "custom_id": manifest_df.iloc[0]["custom_id"],
    "response": {
        "status_code": 200,
        "body": {
            "id": "resp_smoke_test",
            "output_text": json.dumps({"category": "NEUTRAL", "evidence": ""}),
        },
    },
}

with tempfile.TemporaryDirectory() as tmpdir:
    smoke_output_path = Path(tmpdir) / "smoke_output.jsonl"
    smoke_output_path.write_text(json.dumps(smoke_record) + "\n", encoding="utf-8")
    parsed_results_df, parsed_errors_df = parse_batch_output_file(
        smoke_output_path,
        manifest_df,
        default_model_name=MODEL_NAME,
    )

if not parsed_errors_df.empty:
    raise AssertionError("The parser smoke test returned errors.")
if parsed_results_df.columns.tolist() != LEGACY_RESULT_COLUMNS:
    raise AssertionError(parsed_results_df.columns.tolist())

print(f"Batch requests ready: {len(batch_requests_df):,}")
display(manifest_df.head(5))
display(parsed_results_df.head(1))


Batch requests ready: 11,975


,custom_id,hit_id,row_id,source,Title,hit_text,context_idx,count_hits,count_unique_entities,context_window
0,media-frame-d844fe7dbcb4b28a,d844fe7dbcb4b28a,1,Antispiegel,Bereitet der Westen die Entmachtung oder sogar...,Politico,1,1,1,"Er hat es geschafft, seine Leute überall zu pl..."
1,media-frame-f785774b2205e418,f785774b2205e418,4,Antispiegel,Fordert Russland wirklich die Vernichtung alle...,Bild-Zeitung,1,1,1,Viele haben Reitschuster aus der Corona-Zeit j...
2,media-frame-8beba1b1263837c4,8beba1b1263837c4,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,1,1,1,Der Spiegel macht mal wieder Berichterstattung...
3,media-frame-db28e215f2d3989d,db28e215f2d3989d,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,2,1,1,Das NABU ist das vielleicht wichtigste Instrum...
4,media-frame-387a8dd7ac2cedde,387a8dd7ac2cedde,5,Antispiegel,Der Spiegel macht mal wieder Berichterstattung...,Spiegel,3,4,1,"Ich denke, die Antwort liegt auf der Hand. Mär..."


,hit_id,row_id,source,Title,hit_text,context_idx,count_hits,count_unique_entities,context_window,model,response_id,category,evidence,raw_response_json
0,d844fe7dbcb4b28a,1,Antispiegel,Bereitet der Westen die Entmachtung oder sogar...,Politico,1,1,1,"Er hat es geschafft, seine Leute überall zu pl...",gpt-5-mini,resp_smoke_test,NEUTRAL,,"{""id"": ""resp_smoke_test"", ""output_text"": ""{\""c..."


## 4. Write the Final Files and Estimate Batch Cost

The pricing constants below are easy to update if OpenAI changes pricing later.

Note on split sizes:
The split parts can look very uneven in number of rows. That is expected because the notebook splits by estimated input tokens, not by request count. Context windows vary a lot in length, so `part01` is filled until it reaches the safer token threshold and `part02` contains the remaining rows.


In [12]:
write_batch_jsonl(batch_requests_df, FULL_BATCH_JSONL_PATH)
write_manifest_csv(batch_requests_df, FULL_MANIFEST_PATH)

batch_parts = split_batch_requests_by_estimated_tokens(
    batch_requests_df,
    max_estimated_input_tokens=SAFE_BATCH_INPUT_TOKEN_LIMIT,
)
PART_JSONL_PATHS = []
PART_MANIFEST_PATHS = []
for idx, part_df in enumerate(batch_parts, start=1):
    part_jsonl_path = OUTPUT_DIR / f"media_framing_thesis_batch_part{idx:02d}.jsonl"
    part_manifest_path = OUTPUT_DIR / f"media_framing_thesis_manifest_part{idx:02d}.csv"
    write_batch_jsonl(part_df, part_jsonl_path)
    write_manifest_csv(part_df, part_manifest_path)
    PART_JSONL_PATHS.append(part_jsonl_path)
    PART_MANIFEST_PATHS.append(part_manifest_path)

BATCH_INPUT_PRICE_PER_MILLION = 0.25
BATCH_OUTPUT_PRICE_PER_MILLION = 2.00
OUTPUT_TOKEN_SCENARIOS = [40, 80]

cost_rows = []
for output_tokens_per_request in OUTPUT_TOKEN_SCENARIOS:
    estimate = estimate_batch_cost(
        batch_requests_df,
        output_tokens_per_request=output_tokens_per_request,
        batch_input_price_per_million=BATCH_INPUT_PRICE_PER_MILLION,
        batch_output_price_per_million=BATCH_OUTPUT_PRICE_PER_MILLION,
    )
    cost_rows.append(
        {
            "batch_label": "full",
            "assumed_output_tokens_per_request": output_tokens_per_request,
            "requests": estimate.n_requests,
            "estimated_input_tokens": estimate.estimated_input_tokens,
            "estimated_output_tokens": estimate.estimated_output_tokens,
            "estimated_total_cost_usd": round(estimate.estimated_total_cost_usd, 4),
        }
    )

    for part_idx, part_df in enumerate(batch_parts, start=1):
        part_estimate = estimate_batch_cost(
            part_df,
            output_tokens_per_request=output_tokens_per_request,
            batch_input_price_per_million=BATCH_INPUT_PRICE_PER_MILLION,
            batch_output_price_per_million=BATCH_OUTPUT_PRICE_PER_MILLION,
        )
        cost_rows.append(
            {
                "batch_label": f"part{part_idx:02d}",
                "assumed_output_tokens_per_request": output_tokens_per_request,
                "requests": part_estimate.n_requests,
                "estimated_input_tokens": part_estimate.estimated_input_tokens,
                "estimated_output_tokens": part_estimate.estimated_output_tokens,
                "estimated_total_cost_usd": round(part_estimate.estimated_total_cost_usd, 4),
            }
        )

cost_estimate_df = pd.DataFrame(cost_rows)
UPLOAD_BATCH_JSONL_PATH = PART_JSONL_PATHS[0] if PART_JSONL_PATHS else FULL_BATCH_JSONL_PATH
UPLOAD_MANIFEST_PATH = PART_MANIFEST_PATHS[0] if PART_MANIFEST_PATHS else FULL_MANIFEST_PATH

print(f"Batch JSONL written to: {FULL_BATCH_JSONL_PATH}")
print(f"Manifest written to: {FULL_MANIFEST_PATH}")
print(f"Safe split threshold (estimated input tokens): {SAFE_BATCH_INPUT_TOKEN_LIMIT:,}")
print(f"Split batch parts written: {len(PART_JSONL_PATHS):,}")
print(f"Default upload target: {UPLOAD_BATCH_JSONL_PATH}")
print("This notebook now defaults to a Tier-1-safe split for gpt-5-mini. Increase SAFE_BATCH_INPUT_TOKEN_LIMIT only if your OpenAI org has a higher batch queue limit.")
display(cost_estimate_df)


Batch JSONL written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_batch.jsonl
Manifest written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_manifest.csv
Safe split threshold (estimated input tokens): 4,500,000
Split batch parts written: 5
Default upload target: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_batch_part01.jsonl
This notebook now defaults to a Tier-1-safe split for gpt-5-mini. Increase SAFE_BATCH_INPUT_TOKEN_LIMIT only if your OpenAI org has a higher batch queue limit.


,batch_label,assumed_output_tokens_per_request,requests,estimated_input_tokens,estimated_output_tokens,estimated_total_cost_usd
0,full,40,11975,20266739,479000,6.0247
1,part01,40,2609,4498692,104360,1.3334
2,part02,40,2608,4499392,104320,1.3335
3,part03,40,2657,4499582,106280,1.3375
4,part04,40,2727,4498776,109080,1.3429
5,part05,40,1374,2270297,54960,0.6775
6,full,80,11975,20266739,958000,6.9827
7,part01,80,2609,4498692,208720,1.5421
8,part02,80,2608,4499392,208640,1.5421
9,part03,80,2657,4499582,212560,1.5500


## 5. Optional: Upload the Final JSONL and Create the Batch Job

This is intentionally disabled by default so the notebook cannot submit anything accidentally.


In [13]:
RUN_UPLOAD_AND_CREATE_BATCH = True

if RUN_UPLOAD_AND_CREATE_BATCH:
    api_key, api_key_source = read_env_value("OPENAI_API_KEY", project_root=PROJECT_ROOT)
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not found. Add it to .env or export it in your shell.")

    upload_response = upload_batch_file(api_key, UPLOAD_BATCH_JSONL_PATH)
    batch_response = create_batch_job(
        api_key,
        input_file_id=upload_response["id"],
        metadata={
            "project": "thesis_media_framing",
            "manifest": UPLOAD_MANIFEST_PATH.name,
            "model": MODEL_NAME,
        },
    )
    BATCH_JOB_PATH.write_text(json.dumps(batch_response, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"API key source: {api_key_source}")
    print(f"Uploaded JSONL path: {UPLOAD_BATCH_JSONL_PATH}")
    print(f"Uploaded input file id: {upload_response['id']}")
    print(f"Created batch id: {batch_response['id']}")
    print(f"Batch job metadata written to: {BATCH_JOB_PATH}")
else:
    print(
        "Upload disabled. Review the manifest and JSONL first, then set RUN_UPLOAD_AND_CREATE_BATCH = True only when you really want to submit the final batch."
    )


API key source: environment variable
Uploaded JSONL path: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_batch_part01.jsonl
Uploaded input file id: file-ThJks9Bo5tgn7wUhTQ287j
Created batch id: batch_69c45bbaebe08190a750b26c713dd3e1
Batch job metadata written to: /Users/MattisHaumann/Dev/Thesis/2a_NER/outputs/batch_media_framing/final_thesis/media_framing_thesis_batch_job.json


## 6. Optional: Download and Parse the Batch Results Later

Once the batch is completed, fill in the file IDs and rerun this cell. The parsed output is written back into the legacy result-row shape.


In [14]:
DOWNLOAD_AND_PARSE_RESULTS = False
OUTPUT_FILE_ID = ""
ERROR_FILE_ID = ""

if DOWNLOAD_AND_PARSE_RESULTS:
    api_key, api_key_source = read_env_value("OPENAI_API_KEY", project_root=PROJECT_ROOT)
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not found. Add it to .env or export it in your shell.")
    if not OUTPUT_FILE_ID:
        raise ValueError("Set OUTPUT_FILE_ID before downloading batch results.")

    downloaded_output_path = download_openai_file(
        api_key,
        OUTPUT_FILE_ID,
        OUTPUT_DIR / "media_framing_thesis_output.jsonl",
    )
    results_df, errors_df = parse_batch_output_file(
        downloaded_output_path,
        pd.read_csv(FULL_MANIFEST_PATH),
        default_model_name=MODEL_NAME,
    )
    results_df = results_df[LEGACY_RESULT_COLUMNS]
    results_df.to_csv(FULL_RESULTS_PATH, index=False, encoding="utf-8")
    errors_df.to_csv(FULL_ERRORS_PATH, index=False, encoding="utf-8")

    if ERROR_FILE_ID:
        download_openai_file(
            api_key,
            ERROR_FILE_ID,
            OUTPUT_DIR / "media_framing_thesis_error.jsonl",
        )

    print(f"API key source: {api_key_source}")
    print(f"Parsed results written to: {FULL_RESULTS_PATH}")
    print(f"Parsed errors written to: {FULL_ERRORS_PATH}")
    display(results_df.head(5))
    display(errors_df.head(5))
else:
    print(
        "Parsing disabled. Fill OUTPUT_FILE_ID later and set DOWNLOAD_AND_PARSE_RESULTS = True after the batch has completed."
    )


Parsing disabled. Fill OUTPUT_FILE_ID later and set DOWNLOAD_AND_PARSE_RESULTS = True after the batch has completed.
